In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, metrics
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# --- 1. Configuration & Hyperparameters ---
SAMPLE_RATE           = 16000
CLIP_DURATION_MS      = 2000
WINDOW_SIZE_MS        = 30.0
WINDOW_STRIDE_MS      = 20.0
DCT_COEFFICIENT_COUNT = 40
BATCH_SIZE            = 32
EPOCHS                = 30
LEARNING_RATE         = 0.0005

DESIRED_SAMPLES        = int(SAMPLE_RATE * CLIP_DURATION_MS / 1000)
WINDOW_SIZE_SAMPLES    = int(SAMPLE_RATE * WINDOW_SIZE_MS / 1000)
WINDOW_STRIDE_SAMPLES  = int(SAMPLE_RATE * WINDOW_STRIDE_MS / 1000)
SPECTROGRAM_LENGTH     = 1 + int((DESIRED_SAMPLES - WINDOW_SIZE_SAMPLES) / WINDOW_STRIDE_SAMPLES)

print("DESIRED_SAMPLES:", DESIRED_SAMPLES)
print("SPECTROGRAM_LENGTH:", SPECTROGRAM_LENGTH)
print("DCT_COEFFICIENT_COUNT:", DCT_COEFFICIENT_COUNT)

DESIRED_SAMPLES: 32000
SPECTROGRAM_LENGTH: 99
DCT_COEFFICIENT_COUNT: 40


In [3]:
def load_and_label_npy_files():
    """Loads file paths and assigns labels based on the directory."""
    norm_files   = glob.glob(os.path.expanduser('~/npy/npy_tts/Non_Distress/*.npy'))
    abnorm_files = glob.glob(os.path.expanduser('~/npy/npy_tts/Distress/*.npy'))
    file_paths   = norm_files + abnorm_files
    labels       = [0] * len(norm_files) + [1] * len(abnorm_files)
    print(f"Normal files: {len(norm_files)}, Abnormal files: {len(abnorm_files)}")
    return file_paths, labels

def augment_audio(audio):
    """Applies random augmentation to make the model robust to mic noise."""
    # Random background noise (simulates SensorTail mic noise floor)
    noise_level = tf.random.uniform([], 0.001, 0.008)
    noise       = tf.random.normal(shape=tf.shape(audio), stddev=noise_level)
    audio       = audio + noise

    # Random volume shift ±15% (simulates different speaker distances)
    gain        = tf.random.uniform([], 0.85, 1.15)
    audio       = audio * gain

    # Clip to valid float range to avoid STFT instability
    audio       = tf.clip_by_value(audio, -1.0, 1.0)
    return audio

def preprocess_audio(audio_raw, training=False):
    """
    Standardizes audio length and computes MFCCs.
    Follows the logic from your get_preprocess_audio_func.
    """
    # Ensure audio is float32 and normalized
    audio = tf.cast(audio_raw, tf.float32)

    # Pad or Clip to 1 second (16000 samples)
    audio = audio / (tf.reduce_max(tf.abs(audio)) + 1e-6)

    if training:
        audio = augment_audio(audio)

    pad_amount = tf.maximum(0, DESIRED_SAMPLES - tf.shape(audio)[0])
    audio      = tf.pad(audio, [[0, pad_amount]])
    audio      = audio[:DESIRED_SAMPLES]

    #Compute STFT
    stfts = tf.signal.stft(
        audio,
        frame_length=WINDOW_SIZE_SAMPLES,
        frame_step=WINDOW_STRIDE_SAMPLES,
        fft_length=None,
        window_fn=tf.signal.hann_window
    )
    spectrograms        = tf.abs(stfts)
    num_spectrogram_bins = spectrograms.shape[-1]

    #Mel weights
    linear_to_mel = tf.signal.linear_to_mel_weight_matrix(
        40, num_spectrogram_bins, SAMPLE_RATE, 20.0, 4000.0
    )
    mel_spectrograms     = tf.tensordot(spectrograms, linear_to_mel, 1)
    log_mel_spectrograms = tf.math.log(mel_spectrograms + 1e-6)

    #MFCCs
    mfccs = tf.signal.mfccs_from_log_mel_spectrograms(
        log_mel_spectrograms
    )[..., :DCT_COEFFICIENT_COUNT]

    mfccs = tf.expand_dims(mfccs, axis=-1)
    return mfccs


def npy_generator(file_paths, labels, training=False):
    """Generator to yield preprocessed audio and labels."""
    for path, label in zip(file_paths, labels):
        try:
            data       = np.load(path)
            # User specified: first column is audio data
            audio_data = data[:, 0]
            features   = preprocess_audio(audio_data, training=training)
            features   = features.numpy()
            yield features, label
        except Exception as e:
            print(f"Error loading {path}: {e}")
            continue

In [4]:
# --- 3. Model Definition (Inspired by your DS_CNN) ---
def get_distress_model():
    input_shape = (SPECTROGRAM_LENGTH, DCT_COEFFICIENT_COUNT, 1)
    filters     = 64

    inputs = layers.Input(shape=input_shape)

    #Initial Conv
    x = layers.Conv2D(filters, (10, 4), strides=(2, 2), padding='same',kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.2)(x)

    # Deapthwise Separable Block 1
    x = layers.DepthwiseConv2D(kernel_size=(3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (1, 1), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Deapthwise Separable Block 2
    x = layers.DepthwiseConv2D(kernel_size=(3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (1, 1), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Deapthwise Separable Block 3
    x = layers.DepthwiseConv2D(kernel_size=(3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters * 2, (1, 1), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Global Pooling and Output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    # Binary classification: use 1 unit with sigmoid OR 2 with softmax
    # Using 2 units with Softmax to match your sparse_categorical_crossentropy style
    outputs = layers.Dense(2, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=[metrics.SparseCategoricalAccuracy()]
    )
    return model

In [5]:
file_paths, labels = load_and_label_npy_files()

train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

class_weights_array = compute_class_weight(
    'balanced',
    classes=np.array([0, 1]),
    y=train_labels
)
class_weight_dict = {0: float(class_weights_array[0]), 1: float(class_weights_array[1])}
print("Class weights:", class_weight_dict)
print("Train samples:", len(train_paths))
print("Val samples:  ", len(val_paths))

Normal files: 11615, Abnormal files: 11615
Class weights: {0: 1.0, 1: 1.0}
Train samples: 18584
Val samples:   4646


In [6]:
output_signature = (
    tf.TensorSpec(shape=(SPECTROGRAM_LENGTH, DCT_COEFFICIENT_COUNT, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int32)
)

# Calculate steps per epoch for the fit method
train_steps = len(train_paths) // BATCH_SIZE
val_steps   = len(val_paths)   // BATCH_SIZE

print("train_steps:", train_steps)
print("val_steps:  ", val_steps)

train_ds = tf.data.Dataset.from_generator(
    lambda: npy_generator(train_paths, train_labels, training=True),
    output_signature=output_signature
)
train_ds = train_ds.repeat().shuffle(len(train_paths)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    lambda: npy_generator(val_paths, val_labels, training=False),
    output_signature=output_signature
)
val_ds = val_ds.repeat().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Verify shapes before training
print("\nVerifying dataset shapes...")
for feat, label in train_ds.take(1):
    print("Train batch feature shape:", feat.shape)
    print("Train batch label shape:  ", label.shape)

for feat, label in val_ds.take(1):
    print("Val batch feature shape:  ", feat.shape)
    print("Val batch label shape:    ", label.shape)

train_steps: 580
val_steps:   145

Verifying dataset shapes...


E0000 00:00:1778591442.680741  592920 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0000 00:00:1778591442.811105  593038 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1778591452.813857  593038 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 344 of 18584
I0000 00:00:1778591462.820173  593038 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 703 of 18584
I0000 00:00:1778591482.814842  593038 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 1462 of 18584
I0000 00:00:1778591492.816946  593038 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 1840 of 18584
I0000 00:00:1778591512.806817  593038 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while

Train batch feature shape: (32, 99, 40, 1)
Train batch label shape:   (32,)
Val batch feature shape:   (32, 99, 40, 1)
Val batch label shape:     (32,)


In [7]:
# Updated training Cell
model = get_distress_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 99, 40, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 50, 20, 64)     │         2,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 50, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 50, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 50, 20, 64)     │           640 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 50, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 50, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 50, 20, 64)     │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 50, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 50, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 50, 20, 64)     │           640 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 50, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 50, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 50, 20, 64)     │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 50, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 50, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (None, 50, 20, 64)     │           640 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 50, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 50, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 23,490 (91.76 KB)

 Trainable params: 22,466 (87.76 KB)

 Non-trainable params: 1,024 (4.00 KB)

In [8]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4,
        min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=8,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'model_USeng_tts.h5',
        monitor='val_loss', save_best_only=True, verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

Epoch 1/30


I0000 00:00:1778591987.377818  597012 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 224 of 18584
I0000 00:00:1778591997.395996  597012 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 577 of 18584
I0000 00:00:1778592017.384675  597012 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 1247 of 18584
I0000 00:00:1778592027.390025  597012 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 1610 of 18584
I0000 00:00:1778592047.385576  597012 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 2270 of 18584
I0000 00:00:1778592057.397156  597012 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffle buffer (this may take a while): 2552 of 18584
I0000 00:00:1778592067.398527  597012 shuffle_dataset_op.cc:453] ShuffleDatasetV3:3: Filling up shuffl

580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.4139 - sparse_categorical_accuracy: 0.7928
Epoch 1: val_loss improved from None to 0.07863, saving model to model_USeng_tts.h5



Epoch 1: finished saving model to model_USeng_tts.h5
580/580 ━━━━━━━━━━━━━━━━━━━━ 1370s 1s/step - loss: 0.2176 - sparse_categorical_accuracy: 0.9163 - val_loss: 0.0786 - val_sparse_categorical_accuracy: 0.9950 - learning_rate: 5.0000e-04
Epoch 2/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0371 - sparse_categorical_accuracy: 0.9952
Epoch 2: val_loss did not improve from 0.07863
580/580 ━━━━━━━━━━━━━━━━━━━━ 783s 1s/step - loss: 0.0303 - sparse_categorical_accuracy: 0.9960 - val_loss: 0.2873 - val_sparse_categorical_accuracy: 0.8752 - learning_rate: 5.0000e-04
Epoch 3/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0174 - sparse_categorical_accuracy: 0.9976
Epoch 3: val_loss improved from 0.07863 to 0.02514, saving model to model_USeng_tts.h5



Epoch 3: finished saving model to model_USeng_tts.h5
580/580 ━━━━━━━━━━━━━━━━━━━━ 786s 1s/step - loss: 0.0155 - sparse_categorical_accuracy: 0.9977 - val_loss: 0.0251 - val_sparse_categorical_accuracy: 0.9901 - learning_rate: 5.0000e-04
Epoch 4/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0106 - sparse_categorical_accuracy: 0.9983
Epoch 4: val_loss improved from 0.02514 to 0.01413, saving model to model_USeng_tts.h5



Epoch 4: finished saving model to model_USeng_tts.h5
580/580 ━━━━━━━━━━━━━━━━━━━━ 800s 1s/step - loss: 0.0096 - sparse_categorical_accuracy: 0.9987 - val_loss: 0.0141 - val_sparse_categorical_accuracy: 0.9970 - learning_rate: 5.0000e-04
Epoch 5/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0106 - sparse_categorical_accuracy: 0.9979
Epoch 5: val_loss did not improve from 0.01413
580/580 ━━━━━━━━━━━━━━━━━━━━ 806s 1s/step - loss: 0.0096 - sparse_categorical_accuracy: 0.9981 - val_loss: 0.2493 - val_sparse_categorical_accuracy: 0.9071 - learning_rate: 5.0000e-04
Epoch 6/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0050 - sparse_categorical_accuracy: 0.9994
Epoch 6: val_loss did not improve from 0.01413
580/580 ━━━━━━━━━━━━━━━━━━━━ 836s 1s/step - loss: 0.0047 - sparse_categorical_accuracy: 0.9994 - val_loss: 0.0348 - val_sparse_categorical_accuracy: 0.9873 - learning_rate: 5.0000e-04
Epoch 7/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 946ms/step - loss: 0.0053 - sparse_categorical_ac


Epoch 9: finished saving model to model_USeng_tts.h5
580/580 ━━━━━━━━━━━━━━━━━━━━ 789s 1s/step - loss: 0.0032 - sparse_categorical_accuracy: 0.9995 - val_loss: 0.0098 - val_sparse_categorical_accuracy: 0.9974 - learning_rate: 2.5000e-04
Epoch 10/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 945ms/step - loss: 0.0021 - sparse_categorical_accuracy: 0.9998
Epoch 10: val_loss improved from 0.00984 to 0.00332, saving model to model_USeng_tts.h5



Epoch 10: finished saving model to model_USeng_tts.h5
580/580 ━━━━━━━━━━━━━━━━━━━━ 803s 1s/step - loss: 0.0021 - sparse_categorical_accuracy: 0.9998 - val_loss: 0.0033 - val_sparse_categorical_accuracy: 0.9994 - learning_rate: 2.5000e-04
Epoch 11/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 910ms/step - loss: 0.0035 - sparse_categorical_accuracy: 0.9991
Epoch 11: val_loss did not improve from 0.00332
580/580 ━━━━━━━━━━━━━━━━━━━━ 802s 1s/step - loss: 0.0040 - sparse_categorical_accuracy: 0.9989 - val_loss: 0.0037 - val_sparse_categorical_accuracy: 0.9994 - learning_rate: 2.5000e-04
Epoch 12/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 874ms/step - loss: 0.0016 - sparse_categorical_accuracy: 0.9999
Epoch 12: val_loss did not improve from 0.00332
580/580 ━━━━━━━━━━━━━━━━━━━━ 802s 1s/step - loss: 0.0020 - sparse_categorical_accuracy: 0.9998 - val_loss: 0.1384 - val_sparse_categorical_accuracy: 0.9543 - learning_rate: 2.5000e-04
Epoch 13/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 853ms/step - loss: 0.0016 - sparse_ca


Epoch 14: finished saving model to model_USeng_tts.h5
580/580 ━━━━━━━━━━━━━━━━━━━━ 802s 1s/step - loss: 0.0024 - sparse_categorical_accuracy: 0.9995 - val_loss: 0.0025 - val_sparse_categorical_accuracy: 0.9994 - learning_rate: 2.5000e-04
Epoch 15/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 903ms/step - loss: 0.0027 - sparse_categorical_accuracy: 0.9992
Epoch 15: val_loss did not improve from 0.00246
580/580 ━━━━━━━━━━━━━━━━━━━━ 758s 1s/step - loss: 0.0024 - sparse_categorical_accuracy: 0.9994 - val_loss: 0.0875 - val_sparse_categorical_accuracy: 0.9700 - learning_rate: 2.5000e-04
Epoch 16/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 941ms/step - loss: 0.0032 - sparse_categorical_accuracy: 0.9992
Epoch 16: val_loss did not improve from 0.00246
580/580 ━━━━━━━━━━━━━━━━━━━━ 801s 1s/step - loss: 0.0024 - sparse_categorical_accuracy: 0.9995 - val_loss: 0.0155 - val_sparse_categorical_accuracy: 0.9940 - learning_rate: 2.5000e-04
Epoch 17/30
580/580 ━━━━━━━━━━━━━━━━━━━━ 0s 899ms/step - loss: 0.0011 - sparse_ca

In [9]:
model = tf.keras.models.load_model('model_USeng_tts.h5')

val_results = model.evaluate(val_ds, steps=val_steps, return_dict=True)
print(f"Validation Loss:     {val_results['loss']:.4f}")
print(f"Validation Accuracy: {val_results['sparse_categorical_accuracy']:.4f}")

145/145 ━━━━━━━━━━━━━━━━━━━━ 114s 787ms/step - loss: 0.0025 - sparse_categorical_accuracy: 0.9994
Validation Loss:     0.0025
Validation Accuracy: 0.9994
